# TTM_GEMMA2 OOD256 — 12 frozen checkpoints, inference-only one-shot (11/11/2026)

**Phê duyệt owner:** một lượt chấm 3 seed × 4 arm, không train/resume/finetune, không chọn checkpoint/corpus, không GGUF/application; kết quả khoa học phân xử độc lập, L0020 bất biến.

Bộ dữ liệu được niêm phong trước: 256 bài, 512 cửa sổ, 131.072 nhãn, byte SHA bắt buộc. Các checkpoint `step_2048` của SEMANTIC4X3 nằm trong Google Drive private và chỉ **đọc**.

**Chạy Run all một lần** trên Google Colab GPU T4 (ưu tiên Free). Notebook sẽ yêu cầu chọn đúng `OOD256_COLAB_PRIVATE_TRANSFER.zip` từ máy Windows nếu bộ dữ liệu chưa có trong Drive. Mỗi cổng CPU/source/checkpoint SHA phải PASS *trước* tạo `CLAIM.json` và trước forward. Bất kỳ lỗi nào: HOLD, không tự khởi chạy lại hoặc thay dữ liệu.

Không gửi ZIP, private membership, token windows hoặc per-document loss lên GitHub. Toàn bộ tệp riêng tư và receipts được giữ trong `MyDrive/TTM/ttm-small-pilot/ttm_gemma2_ood256_12arm_inference_only_20261111_v1`.

In [ ]:
from pathlib import Path
import hashlib, io, os, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/TTM/ttm-small-pilot")
STUDY=ROOT/"ttm_gemma2_semantic4x3_3seed_2048x4096_h768_20261110_v2"
SOURCE_ZIP=ROOT/"TTM_GEMMA2_CAUSAL6_INPUT_PRIVATE_V1_20261109.zip"
WORK=ROOT/"ttm_gemma2_ood256_12arm_inference_only_20261111_v1"
OOD=WORK/"input_ood"
OUT=WORK/"science_one_shot"
SCRIPT_DIR=WORK/"frozen_source"

assert ROOT.is_dir() and STUDY.is_dir(), "HOLD_DRIVE_OR_FROZEN_STUDY_MISSING"
assert SOURCE_ZIP.is_file(), "HOLD_PRIVATE_FROZEN_SOURCE_ZIP_MISSING"
WORK.mkdir(exist_ok=True)
OOD.mkdir(exist_ok=True)
SCRIPT_DIR.mkdir(exist_ok=True)

REQUIRED={
  "OOD256_PUBLIC_SEAL.json":"0091b56b27c68f1bb724b63181097434d6c7c5922e5cc1de929423b5b0c45438",
  "OOD256_PRIVATE_MEMBERSHIP.json":"627c99db8b39b6479fdb074a82b0ab7eea0367dde49bf0072c29ff410ecdcd5f",
  "OOD256_PRIVATE_WINDOWS.npz":"3ec7655c512c5d13d4c0d04d1ddbae80b8ea0106aee82064e8dd806786d8336a"
}
def sha_bytes(x):return hashlib.sha256(x).hexdigest()
def sha_file(p):
  h=hashlib.sha256()
  with open(p,"rb") as f:
    for b in iter(lambda:f.read(1<<20),b""):h.update(b)
  return h.hexdigest()

if not all((OOD/k).is_file() and sha_file(OOD/k)==v for k,v in REQUIRED.items()):
  assert not any((OOD/k).exists() for k in REQUIRED), "HOLD_PARTIAL_OOD_STAGE_NO_OVERWRITE"
  print("Select OOD256_COLAB_PRIVATE_TRANSFER.zip built on Windows (3 sealed files, NO raw WikiText).")
  uploaded=files.upload()
  assert set(uploaded)=={"OOD256_COLAB_PRIVATE_TRANSFER.zip"},"HOLD_SINGLE_ZIP_REQUIRED"
  with zipfile.ZipFile(io.BytesIO(uploaded["OOD256_COLAB_PRIVATE_TRANSFER.zip"])) as z:
    assert set(z.namelist())==set(REQUIRED),"HOLD_ZIP_MEMBER_SET"
    contents={name:z.read(name) for name in REQUIRED}
  assert all(sha_bytes(contents[k])==v for k,v in REQUIRED.items()),"HOLD_OOD_PRIVATE_TRANSFER_SHA"
  for name in REQUIRED:
    with (OOD/name).open("xb") as f:f.write(contents[name])
assert all(sha_file(OOD/k)==v for k,v in REQUIRED.items()),"HOLD_OOD_PRIVATE_SHA"
assert not OUT.exists(),"HOLD_SCIENCE_OUTPUT_ALREADY_PRESENT_NO_SECOND_CLAIM"
print("OOD256_PRIVATE_TRANSFER=PASS_EXACT_3_SHA; STUDY=FROZEN; OWNER_PERMIT=ONE_SHOT")

In [ ]:
import os,sys,subprocess,urllib.request,hashlib
SOURCES=[
 ("ttm_ood256_inference_only_12arm.py",
  "44721a6ac0d206756a0e45db5fd36f60cb177fd6",
  "7f6cd35a149569016391128ea6e048ad2930cc80"),
 ("ttm_ood256_independent_science_readback.py",
  "3f49d16a98d008f2ddb976ce0f8e4d316ce7b2e2",
  "2c278a1e8ce3597307e988a866178981cc81f5f1"),
 ("ttm_ood256_scorer_interface_cpu_fixture.py",
  "6673f6007f114f40dd1e8f846ad1ef42673716f5",
  "0ec741687827ba85e4f435d83a9b12207894ed07")
]
BASE="https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"
SUB="tools/embeddinggemma2_swap/ood_wikitext2_v1/"
for name,commit,blob in SOURCES:
  p=SCRIPT_DIR/name
  if not p.exists():
    url=BASE+commit+"/"+SUB+name
    with urllib.request.urlopen(url,timeout=90) as response:
      data=response.read()
    digest=hashlib.sha1(b"blob "+str(len(data)).encode()+b"\0"+data).hexdigest()
    assert digest==blob, "HOLD_SOURCE_GIT_BLOB_DRIFT_"+name
    with p.open("xb") as f:f.write(data)
  data=p.read_bytes()
  digest=hashlib.sha1(b"blob "+str(len(data)).encode()+b"\0"+data).hexdigest()
  assert digest==blob, "HOLD_SOURCE_GIT_BLOB_DRIFT_"+name
  print(name, "GIT_BLOB_SHA1=PASS", "SHA256="+hashlib.sha256(data).hexdigest())

env=os.environ.copy()
env["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
env["PYTHONDONTWRITEBYTECODE"]="1"
PY=sys.executable
def run(args):
  print("EXECUTE:",args[1:4],flush=True)
  subprocess.run(args,check=True,env=env)
runner=SCRIPT_DIR/"ttm_ood256_inference_only_12arm.py"
checker=SCRIPT_DIR/"ttm_ood256_independent_science_readback.py"
fixture=SCRIPT_DIR/"ttm_ood256_scorer_interface_cpu_fixture.py"

# Synthetic QA only. Neither test loads checkpoints nor calls model.forward.
run([PY,str(runner),"--mode","fixture"])
run([PY,str(checker),"--mode","fixture"])
run([PY,str(fixture),"--mode","fixture"])
print("OOD12_SOURCE_SYNTHETIC_QA=PASS_ALL; SCIENCE=NOT_YET_EXECUTED")

In [ ]:
# Single integrated preflight + owner-authorized inference-only execution.
# This script verifies all checkpoint/READY/source/3 OOD SHA and model construction
# BEFORE it creates CLAIM.json and BEFORE the first model.forward.
assert not OUT.exists(),"HOLD_SCIENCE_OUTPUT_ALREADY_EXISTS"
args=[PY,str(runner),"--mode","execute",
      "--study",str(STUDY),"--ood",str(OOD),
      "--source-zip",str(SOURCE_ZIP),
      "--work",str(WORK),"--output",str(OUT),"--owner-approved"]
run(args)
assert (OUT/"RESULT.json").is_file(),"HOLD_RESULT_NOT_READY"
print("OOD12_SCIENTIFIC_SCORING_COMPLETED_PENDING_INDEPENDENT_ADJUDICATION")

In [ ]:
# Independent mathematical readback; never modifies L0020 or lineage.md.
run([PY,str(checker),"--mode","review","--output",str(OUT)])
print("OOD12_INDEPENDENT_QA_READY; OWNER_SCIENCE_ADJUDICATION_STILL_SEPARATE")
print("PUBLIC_RESULT:",OUT/"RESULT.json")
print("PUBLIC_READBACK:",OUT/"INDEPENDENT_READBACK_PUBLIC.json")
print("DO_NOT_PUBLISH:",OUT/"PER_DOCUMENT_PRIVATE.json")